# Field And Trajectory Showcase

This notebook uses the shared synthetic magnetic field and trajectory sampler.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch

NOTEBOOK_ROOT = Path.cwd()
if (NOTEBOOK_ROOT / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT))
elif (NOTEBOOK_ROOT / "notebooks" / "shared").exists():
    sys.path.insert(0, str(NOTEBOOK_ROOT / "notebooks"))

from shared.magentic_field import B, DEVICE, DTYPE, EARTH_RADIUS_M, STEP_METERS
from shared.trajectory_sampler import make_trajectories

## Magnetic Field

In [2]:
n_lat, n_lon = 72, 144
lat = np.linspace(-0.5 * math.pi, 0.5 * math.pi, n_lat)
lon = np.linspace(-math.pi, math.pi, n_lon, endpoint=False)
lat_grid, lon_grid = np.meshgrid(lat, lon, indexing="ij")
x_s = np.cos(lat_grid) * np.cos(lon_grid)
y_s = np.cos(lat_grid) * np.sin(lon_grid)
z_s = np.sin(lat_grid)
sphere_pts = np.stack([x_s, y_s, z_s], axis=-1).reshape(-1, 3)

faces_i, faces_j, faces_k = [], [], []
for a in range(n_lat - 1):
    for b in range(n_lon):
        p00 = a * n_lon + b
        p01 = a * n_lon + (b + 1) % n_lon
        p10 = (a + 1) * n_lon + b
        p11 = (a + 1) * n_lon + (b + 1) % n_lon
        faces_i.extend([p00, p01])
        faces_j.extend([p10, p10])
        faces_k.extend([p01, p11])

with torch.no_grad():
    field = B(torch.as_tensor(sphere_pts, device=DEVICE, dtype=DTYPE)).cpu().numpy()

lo = np.quantile(field, 0.02, axis=0)
hi = np.quantile(field, 0.98, axis=0)
field_rgb = np.clip((field - lo) / (hi - lo + 1e-12), 0.0, 1.0)
basis = np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=float)
rgb = np.clip(field_rgb @ basis, 0, 255).astype(int)
vertexcolor = [f"rgb({r},{g},{b})" for r, g, b in rgb]

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=sphere_pts[:, 0],
            y=sphere_pts[:, 1],
            z=sphere_pts[:, 2],
            i=faces_i,
            j=faces_j,
            k=faces_k,
            vertexcolor=vertexcolor,
            flatshading=False,
            lighting=dict(ambient=0.72, diffuse=0.35, specular=0.08),
            name="B component mixture",
            showscale=False,
        )
    ]
)
fig.update_layout(
    title="Synthetic magnetic field components on the unit sphere (red=Bx, green=By, blue=Bz)",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()

## Example Trajectory

In [3]:
with torch.no_grad():
    _, _, _, _, sample_path = make_trajectories(1, step_counts=16, return_path=True)
path_pts = sample_path[0].cpu().numpy()
center = path_pts.mean(axis=0)
center = center / np.linalg.norm(center)
north = np.array([0.0, 0.0, 1.0])
ref = north if abs(center[2]) < 0.9 else np.array([1.0, 0.0, 0.0])
east = np.cross(ref, center)
east = east / np.linalg.norm(east)
north_local = np.cross(center, east)
local_xy_m = EARTH_RADIUS_M * np.column_stack([path_pts @ east, path_pts @ north_local])
span_m = max(np.ptp(local_xy_m[:, 0]), np.ptp(local_xy_m[:, 1]), 8.0 * STEP_METERS)
patch_radius_m = 3.0 * span_m
patch_radius_rad = patch_radius_m / EARTH_RADIUS_M
patch_mask = np.arccos(np.clip(sphere_pts @ center, -1.0, 1.0)) <= patch_radius_rad
patch_idx = np.flatnonzero(patch_mask)
patch_lookup = -np.ones(len(sphere_pts), dtype=int)
patch_lookup[patch_idx] = np.arange(len(patch_idx))
patch_face_mask = (
    patch_mask[np.asarray(faces_i)]
    & patch_mask[np.asarray(faces_j)]
    & patch_mask[np.asarray(faces_k)]
)
patch_i = patch_lookup[np.asarray(faces_i)[patch_face_mask]]
patch_j = patch_lookup[np.asarray(faces_j)[patch_face_mask]]
patch_k = patch_lookup[np.asarray(faces_k)[patch_face_mask]]
patch_pts = sphere_pts[patch_idx]
patch_colors = [vertexcolor[idx] for idx in patch_idx]
camera_eye = 1.000004 * center

fig = go.Figure()
fig.add_trace(
    go.Mesh3d(
        x=patch_pts[:, 0],
        y=patch_pts[:, 1],
        z=patch_pts[:, 2],
        i=patch_i,
        j=patch_j,
        k=patch_k,
        vertexcolor=patch_colors,
        opacity=0.40,
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.25, specular=0.03),
        name="B component mixture",
        showscale=False,
    )
)
fig.add_trace(
    go.Scatter3d(
        x=path_pts[:, 0],
        y=path_pts[:, 1],
        z=path_pts[:, 2],
        mode="lines+markers",
        line=dict(color="black", width=7),
        marker=dict(size=4, color=np.arange(path_pts.shape[0]), colorscale="Inferno"),
        name="16-step walk",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=[path_pts[0, 0], path_pts[-1, 0]],
        y=[path_pts[0, 1], path_pts[-1, 1]],
        z=[path_pts[0, 2], path_pts[-1, 2]],
        mode="markers",
        marker=dict(size=[7, 8], color=["lime", "red"]),
        name="start/end",
    )
)
fig.update_layout(
    title=f"Random biased walk sample: 16 steps x {STEP_METERS:.1f} m, zoomed patch",
    scene=dict(
        aspectmode="data",
        camera=dict(eye=dict(x=camera_eye[0], y=camera_eye[1], z=camera_eye[2])),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()